# LG-03: Human-in-the-loop 人工介入

> **阶段**: LG-03 | **难度**: 中级 | **预计时长**: 3-4 小时

这一章不再用一堆硬编码函数模拟审批。

真实系统里更常见的是：**LLM / Agent 先生成内容、判断下一步、准备调用工具；人只在关键风险点接管。**

本节用一个内容发布场景讲清楚三种落地方式：

| 场景 | 人在哪里介入 | 适合什么时候用 |
|---|---|---|
| Node 级 HITL | 自定义 `StateGraph` 的某个节点里 `interrupt()` | 你需要完整控制 state、router、审计日志 |
| Agent 级 HITL | Agent 准备调用 tools 前 `interrupt_before=["tools"]` | 你用预构建 Agent，但要拦住危险工具调用 |
| Wrapper 级 HITL | 把危险 tool 包一层，在 wrapper 里 `interrupt()` | 多个 Agent/Graph 都会调用同一个危险能力 |

先记住一句话：**HITL 不是让模型少做事，而是让模型做完准备动作后，危险动作必须等人确认。**

![让自动流程等人审批：人会问，图不会](images/human-vs-graph-approval-cn-premium.png)

> 这张图先建立直觉：人看到风险会问人；Graph 默认只会沿着边继续走。Human-in-the-loop 要解决的，就是让流程在关键动作前等人确认。



In [ ]:
# 安装依赖（如未安装请取消注释）
# !pip install -U langgraph langchain langchain-openai pydantic

# 本 notebook 直接读取项目根目录 .env：
# OPENAI_MODEL=deepseek-v4-flash
# OPENAI_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
# OPENAI_API_KEY=你的 key
# OPENAI_TEMPERATURE=0.7

In [ ]:
from hitl_demo_support import *

print_json("当前使用模型", {"model": MODEL_NAME})

## 1. Node 级 HITL：LLM 生成内容，节点里决定要不要等人

这是最白盒的做法。你自己写 `StateGraph`，所以可以控制：

- state 里保存哪些业务字段
- 哪个节点调用 LLM
- 哪个节点触发 `interrupt()`
- 人的决定如何写回 state
- router 如何选择发布或拒绝

这里的心智很简单：

```text
输入发布请求 → LLM 生成文案 → LLM/规则做风险审查 → 高风险暂停给人 → 人回复 → 继续发布或拒绝
```

注意：State 用 Pydantic `BaseModel`，不是 `TypedDict`。这样默认值、字段类型和运行时校验都更清楚。


In [ ]:
class ReviewInput(BaseModel):
    request_id: str


class ReviewState(BaseModel):
    request_id: str = ""
    topic: str = ""
    channel: str = ""
    owner_id: str = ""
    draft_seed: str = ""
    generated_draft: str = ""
    risk_level: Literal["none", "low", "medium", "high"] = "none"
    risk_score: float = 0.0
    risk_reasons: list[str] = Field(default_factory=list)
    needs_human_review: bool = False
    review_decision: str = ""
    reviewer_id: str = ""
    reviewer_comment: str = ""
    revised_draft: str = ""
    publish_status: str = ""
    final_content: str = ""
    audit_log: list[str] = Field(default_factory=list)


def add_log(state: ReviewState, message: str) -> list[str]:
    return state.audit_log + [message]


def load_request(state: ReviewState) -> dict[str, Any]:
    request = find_publish_request(state.request_id)
    return {
        "topic": request["topic"],
        "channel": request["channel"],
        "owner_id": request["owner_id"],
        "draft_seed": request["draft_seed"],
        "audit_log": add_log(state, f"读取请求: {request['request_id']}"),
    }


node_model = chat_model


def generate_draft_with_llm(state: ReviewState) -> dict[str, Any]:
    prompt = f"""
你是一个电商运营文案助手。请根据发布请求生成一版中文营销文案。
只返回 JSON，不要解释。

发布主题：{state.topic}
发布渠道：{state.channel}
原始素材：{state.draft_seed}

JSON 格式：
{{"draft": "...", "reason": "..."}}
""".strip()
    result = parse_json_message(node_model.invoke([HumanMessage(content=prompt)]))
    return {
        "generated_draft": result["draft"],
        "audit_log": add_log(state, f"LLM 生成文案: {result.get('reason', '无说明')}"),
    }


def review_risk_with_llm(state: ReviewState) -> dict[str, Any]:
    prompt = f"""
你是内容合规审核助手。请根据规则判断下面文案是否需要人工审批。
只返回 JSON，不要解释。

审核规则：
{policy_text()}

待审核文案：
{state.generated_draft}

JSON 格式：
{{
  "risk_level": "none|low|medium|high",
  "risk_score": 0.0,
  "risk_reasons": ["..."],
  "needs_human_review": true
}}
""".strip()
    result = parse_json_message(node_model.invoke([HumanMessage(content=prompt)]))
    return {
        "risk_level": result["risk_level"],
        "risk_score": float(result["risk_score"]),
        "risk_reasons": result["risk_reasons"],
        "needs_human_review": bool(result["needs_human_review"]),
        "audit_log": add_log(state, f"LLM 风险审查: {result['risk_level']} / {result['risk_score']}"),
    }


def human_review_node(state: ReviewState) -> dict[str, Any]:
    if not state.needs_human_review:
        return {
            "review_decision": "approve",
            "reviewer_id": "system",
            "reviewer_comment": "风险较低，自动通过。",
            "audit_log": add_log(state, "低风险自动通过"),
        }

    decision = interrupt({
        "action": "content_review_required",
        "request_id": state.request_id,
        "channel": state.channel,
        "draft": state.generated_draft,
        "risk_level": state.risk_level,
        "risk_score": state.risk_score,
        "risk_reasons": state.risk_reasons,
        "options": ["approve", "edit_and_approve", "reject"],
    })
    return {
        "review_decision": decision["decision"],
        "reviewer_id": decision.get("reviewer_id", "reviewer_demo"),
        "reviewer_comment": decision.get("comment", ""),
        "revised_draft": decision.get("revised_draft", ""),
        "audit_log": add_log(state, f"人工审批: {display(decision['decision'])}"),
    }


def route_after_review(state: ReviewState) -> Literal["publish_content", "reject_content"]:
    if state.review_decision == "reject":
        return "reject_content"
    return "publish_content"


def publish_content(state: ReviewState) -> dict[str, Any]:
    final_content = state.revised_draft if state.review_decision == "edit_and_approve" else state.generated_draft
    publish_status = "published_after_edit" if state.review_decision == "edit_and_approve" else "published"
    return {
        "final_content": final_content,
        "publish_status": publish_status,
        "audit_log": add_log(state, f"发布结果: {display(publish_status)}"),
    }


def reject_content(state: ReviewState) -> dict[str, Any]:
    return {
        "final_content": "",
        "publish_status": "rejected",
        "audit_log": add_log(state, "发布结果: 已拒绝"),
    }


node_builder = StateGraph(ReviewState, input_schema=ReviewInput)
node_builder.add_node("load_request", load_request)
node_builder.add_node("generate_draft_with_llm", generate_draft_with_llm)
node_builder.add_node("review_risk_with_llm", review_risk_with_llm)
node_builder.add_node("human_review", human_review_node)
node_builder.add_node("publish_content", publish_content)
node_builder.add_node("reject_content", reject_content)
node_builder.add_edge(START, "load_request")
node_builder.add_edge("load_request", "generate_draft_with_llm")
node_builder.add_edge("generate_draft_with_llm", "review_risk_with_llm")
node_builder.add_edge("review_risk_with_llm", "human_review")
node_builder.add_conditional_edges(
    "human_review",
    route_after_review,
    {"publish_content": "publish_content", "reject_content": "reject_content"},
)
node_builder.add_edge("publish_content", END)
node_builder.add_edge("reject_content", END)

node_graph = node_builder.compile(checkpointer=MemorySaver())
print_json("Node 级 HITL 图已创建", {"model": MODEL_NAME})


### 1.1 第一次执行：输入是发布请求，输出是待审批任务

这段不要从内部字段看，要按产品心智看：

```text
我输入了 request_id
LLM 生成了一版文案
LLM/规则发现风险
Graph 没有发布，而是返回一张待审批任务
```


In [ ]:
node_input = {"request_id": "campaign_high_001"}
node_config = {"configurable": {"thread_id": "node-hitl-demo"}}

print_json("1) 输入给 graph 的内容", node_input)

node_first = node_graph.invoke(node_input, config=node_config)
node_snapshot = node_graph.get_state(node_config)
node_interrupt = node_first["__interrupt__"][0]

approval_task = {
    "任务类型": "内容发布审批",
    "待审核文案": node_interrupt.value["draft"],
    "风险等级": display(node_interrupt.value["risk_level"]),
    "风险分数": node_interrupt.value["risk_score"],
    "风险原因": node_interrupt.value["risk_reasons"],
    "审核员可以选择": node_interrupt.value["options"],
}

print_json("\n2) 第一次输出：不是发布结果，而是待审批任务", approval_task)
print_json("\n3) 现在流程停在哪里", {
    "当前节点": display(node_snapshot.next),
    "流程是否结束": "否",
    "等待谁": "等待审核员提交决定",
})
print_json("\n4) checkpoint 保存的现场摘要", {
    "thread_id": node_config["configurable"]["thread_id"],
    "generated_draft": node_snapshot.values["generated_draft"],
    "risk_level": display(node_snapshot.values["risk_level"]),
    "needs_human_review": display(node_snapshot.values["needs_human_review"]),
})


### 1.2 人处理待办：把决定交回同一个 thread

审核员不是重新发起一次请求，而是把决定交回刚才暂停的那条 thread。

```text
输入：Command(resume=审核决定)
输出：Graph 从 human_review 继续跑，最后发布或拒绝
```


In [ ]:
node_decision = {
    "decision": "edit_and_approve",
    "reviewer_id": "reviewer_demo_001",
    "comment": "删除绝对化和收益承诺表达后发布。",
    "revised_draft": "618 智能空调活动开启，限时下单享专属优惠，具体权益以活动页为准。",
}

print_json("1) 审核员输入的决定", node_decision)

node_final = node_graph.invoke(Command(resume=node_decision), config=node_config)

print_json("\n2) Graph 恢复后的最终输出", {
    "发布状态": display(node_final["publish_status"]),
    "审核决定": display(node_final["review_decision"]),
    "最终发布文案": node_final["final_content"],
})

print("\n3) 审计日志")
for index, item in enumerate(node_final["audit_log"], start=1):
    print(f"{index}. {item}")


## 2. Agent 级 HITL：Agent 想调用工具前，先停下来给人看

上面是自定义 `StateGraph`。但真实项目里，很多人会直接用 Agent：

```text
用户说：帮我发布这条内容
Agent 思考：我需要调用 publish_content 工具
系统暂停：这个工具会真的发布，先给人确认
人确认后：工具才执行
```

这种场景不需要你把审批写进每个节点。你可以让 Agent 在 `tools` 节点前统一暂停：

```python
create_agent(..., interrupt_before=["tools"])
```

注意：这种方式拦的是 **Agent 即将调用工具** 这个动作。人看到的是 tool name 和 tool args。


In [ ]:
@tool
def publish_to_channel(content: str, channel: str) -> str:
    """Publish approved content to a channel. This is a side-effect tool."""
    return json.dumps({
        "status": "published",
        "channel": channel,
        "content": content,
    }, ensure_ascii=False)


agent_model = chat_model


publish_agent = create_agent(
    model=agent_model,
    tools=[publish_to_channel],
    system_prompt="你是内容运营 Agent。只有用户明确要求发布时，才调用 publish_to_channel。",
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],
)

agent_config = {"configurable": {"thread_id": "agent-hitl-demo"}}
agent_input = {
    "messages": [HumanMessage(content="请把这条已经审核过的空调活动文案发布到公众号。")]
}

print_json("Agent 级 HITL 已创建", {"model": MODEL_NAME, "interrupt_before": ["tools"]})
print_json("\n1) 用户输入", {"message": agent_input["messages"][0].content})

agent_first = publish_agent.invoke(agent_input, config=agent_config)
agent_snapshot = publish_agent.get_state(agent_config)
proposed_message = agent_first["messages"][-1]

print_json("\n2) Agent 想做什么", {
    "tool_calls": proposed_message.tool_calls,
})
print_json("\n3) 为什么停住", {
    "当前节点": display(agent_snapshot.next),
    "原因": "tools 节点会执行真实发布动作，所以 interrupt_before=['tools'] 先拦住。",
    "人现在要判断": "允许这个 tool_call 执行吗？",
})


In [ ]:
print_json("1) 人工确认", {
    "decision": "approve",
    "meaning": "允许 Agent 执行刚才的 publish_to_channel 工具调用。",
})

agent_final = publish_agent.invoke(Command(resume={}), config=agent_config)

print("\n2) Agent 恢复后的消息流")
for index, message in enumerate(agent_final["messages"], start=1):
    item = {
        "type": message.type,
        "content": message.content,
    }
    if getattr(message, "tool_calls", None):
        item["tool_calls"] = message.tool_calls
    if getattr(message, "name", None):
        item["tool_name"] = message.name
    print_json(f"message {index}", item)


## 3. Wrapper 级 HITL：把危险工具包起来，谁调用都要审批

Agent 级 `interrupt_before=["tools"]` 很直观，但它是“整批工具节点前暂停”。

生产里还有一种更稳的做法：**危险能力自己带审批**。

比如 `publish_to_channel` 是一个有副作用的能力，那就不要让 Agent 直接调用它，而是暴露一个 wrapper：

```text
Agent 调用 publish_with_human_gate
  → wrapper 先 interrupt，给人看 tool args
  → 人 approve 后，wrapper 才调用真正的 publish service
```

好处是：不管这个工具被哪个 Agent、哪个 Graph 调用，审批逻辑都不会漏。


In [ ]:
def publish_service(content: str, channel: str) -> dict[str, Any]:
    """模拟真实发布服务。生产里这里会调用数据库、CMS、消息队列或外部 API。"""
    return {"status": "published", "channel": channel, "content": content}


@tool
def publish_with_human_gate(content: str, channel: str, reason: str) -> str:
    """Publish content, but ask a human before the side effect happens."""
    decision = interrupt({
        "action": "approve_tool_call",
        "tool": "publish_with_human_gate",
        "channel": channel,
        "content": content,
        "agent_reason": reason,
        "options": ["approve", "edit_and_approve", "reject"],
    })

    if decision["decision"] == "reject":
        return json.dumps({
            "status": "canceled_by_human",
            "reviewer_id": decision.get("reviewer_id", "reviewer_demo"),
            "comment": decision.get("comment", ""),
        }, ensure_ascii=False)

    final_content = decision.get("revised_content", content)
    result = publish_service(final_content, channel)
    result["reviewer_id"] = decision.get("reviewer_id", "reviewer_demo")
    result["human_decision"] = decision["decision"]
    return json.dumps(result, ensure_ascii=False)


wrapper_model = chat_model


wrapper_agent = create_agent(
    model=wrapper_model,
    tools=[publish_with_human_gate],
    system_prompt="你是内容运营 Agent。需要发布时调用 publish_with_human_gate，不要绕过人工确认。",
    checkpointer=MemorySaver(),
)

wrapper_config = {"configurable": {"thread_id": "wrapper-hitl-demo"}}
wrapper_input = {
    "messages": [HumanMessage(content="把会员日活动提醒发布到短信渠道。")]
}

print_json("Wrapper 级 HITL 已创建", {"model": MODEL_NAME})
print_json("\n1) 用户输入", {"message": wrapper_input["messages"][0].content})

wrapper_first = wrapper_agent.invoke(wrapper_input, config=wrapper_config)
wrapper_interrupt = wrapper_first["__interrupt__"][0]
wrapper_snapshot = wrapper_agent.get_state(wrapper_config)

print_json("\n2) Agent 已经决定调用发布工具，但 wrapper 在工具内部暂停", wrapper_interrupt.value)
print_json("\n3) 现在停在哪里", {
    "当前节点": display(wrapper_snapshot.next),
    "心智": "不是 Agent 卡住了，而是危险工具自己要求人工确认。",
})


In [ ]:
wrapper_decision = {
    "decision": "edit_and_approve",
    "reviewer_id": "reviewer_demo_003",
    "comment": "短信里去掉强促销语气，保留活动提醒。",
    "revised_content": "会员日活动开启，预约到店可领取体验礼包，详情以门店活动说明为准。",
}

print_json("1) 人工输入的决定", wrapper_decision)

wrapper_final = wrapper_agent.invoke(Command(resume=wrapper_decision), config=wrapper_config)

print("\n2) Wrapper Agent 恢复后的消息流")
for index, message in enumerate(wrapper_final["messages"], start=1):
    item = {"type": message.type, "content": message.content}
    if getattr(message, "tool_calls", None):
        item["tool_calls"] = message.tool_calls
    if getattr(message, "name", None):
        item["tool_name"] = message.name
    print_json(f"message {index}", item)


## 4. 为什么 checkpointer 还是必须存在

不管你用的是 Node、Agent，还是 wrapper，只要中间要等人，就不能靠 Python 调用栈一直挂着。

真实业务里，审批可能几分钟后回来，也可能第二天才回来。系统通常不会让请求、线程、连接和进程一直占着资源等人。更合理的做法是：

```text
运行到风险点
  → 生成待办
  → checkpoint 保存 thread、暂停点、state/messages
  → 当前请求结束，释放资源
  → 人提交决定
  → Command(resume=...) 从 checkpoint 恢复
```

所以 checkpoint 不是“为了记忆而记忆”。它是在告诉系统：**这个流程可以先断开，之后还能接着跑。**


In [ ]:
print_json("三种 HITL 的区别", {
    "Node 级": {
        "适合": "自定义 StateGraph，需要完整控制 state/router/audit_log",
        "人看到": "业务化审批 payload",
        "恢复方式": "Command(resume=人工决定)",
    },
    "Agent 级": {
        "适合": "预构建 Agent 调工具前统一拦截",
        "人看到": "Agent 准备执行的 tool_calls",
        "恢复方式": "Command(resume={}) 允许 tools 节点继续执行",
    },
    "Wrapper 级": {
        "适合": "危险工具被多个 Agent/Graph 复用",
        "人看到": "wrapper 暴露的业务审批信息",
        "恢复方式": "Command(resume=人工决定)，wrapper 决定是否调用真实服务",
    },
})


## 5. 这节课真正要记住什么

这章不是在讲“怎么暂停代码”。

更准确地说，Human-in-the-loop 是给 Agent 系统加一条安全边界：

```text
模型可以生成、判断、计划、准备 tool_call
但高风险动作真正发生前，必须有人确认
```

三种实现方式按控制粒度选择：

- 你需要完整业务状态和路由：用 **Node 级 HITL**。
- 你已经用了预构建 Agent，只想拦住工具调用：用 **Agent 级 HITL**。
- 你有一个危险工具会被到处复用：用 **Wrapper 级 HITL**。

练习：

1. 把 Node 级例子的人工决定从 `edit_and_approve` 改成 `reject`，观察 router 走向。
2. 给 Agent 级例子再加一个低风险查询工具，比较查询工具和发布工具是否都应该被拦截。
3. 把 wrapper 里的 `publish_service` 换成你自己的 CMS / 数据库 / 消息队列调用。
4. 把 `MemorySaver()` 换成持久化 checkpointer，模拟进程重启后继续恢复审批。
